In [1]:
import sys
import os

spark_home = "/opt/spark"
sys.path.append(os.path.join(spark_home, "python"))
sys.path.append(os.path.join(spark_home, "python/lib/py4j-0.10.9.7-src.zip"))

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SparkSession").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/01 14:27:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
## Where to Look for APIs

df = spark.read.format('csv').option('header','true').option('inferSchema', 'true').load('data/customers.csv')
df.printSchema()
df.show(3)

root
 |-- Index: integer (nullable = true)
 |-- Customer Id: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Phone 1: string (nullable = true)
 |-- Phone 2: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Subscription Date: date (nullable = true)
 |-- Website: string (nullable = true)

+-----+---------------+----------+---------+----------------+--------------+--------+------------------+--------------------+--------------------+-----------------+--------------------+
|Index|    Customer Id|First Name|Last Name|         Company|          City| Country|           Phone 1|             Phone 2|               Email|Subscription Date|             Website|
+-----+---------------+----------+---------+----------------+--------------+--------+------------------+--------------------+------------------

In [4]:
## Working with booleans
from pyspark.sql.functions import col, expr, desc, instr

df.where( col("Index") < 10 ).select("Index", "First Name", "Last Name").show(3)
df.where("Index < 10").select("Index", "First Name", "Last Name").show(3)

indexFilter = (col("Index") < 200) & (col("Index") > 100)
spainFilter = col("Country").contains("ain")
df.where(indexFilter).where(spainFilter).show()

df.where(indexFilter).groupBy("Country").count().sort(expr("count").desc()).show(3)

+-----+----------+---------+
|Index|First Name|Last Name|
+-----+----------+---------+
|    1|     Jared|   Jarvis|
|    2|     Marie|   Malone|
|    3|    Elijah|  Barrera|
+-----+----------+---------+
only showing top 3 rows

+-----+----------+---------+
|Index|First Name|Last Name|
+-----+----------+---------+
|    1|     Jared|   Jarvis|
|    2|     Marie|   Malone|
|    3|    Elijah|  Barrera|
+-----+----------+---------+
only showing top 3 rows



+-----+---------------+----------+---------+--------------------+-----------------+--------------------+--------------------+-------------------+--------------------+-----------------+--------------------+
|Index|    Customer Id|First Name|Last Name|             Company|             City|             Country|             Phone 1|            Phone 2|               Email|Subscription Date|             Website|
+-----+---------------+----------+---------+--------------------+-----------------+--------------------+--------------------+-------------------+--------------------+-----------------+--------------------+
|  109|5aFba4eb0Bc5248|   Maureen|    Weber|          Walton Ltd|       Port Randy|Saint Kitts and N...|        797.490.4550| 930-650-3473x47188|tapiajanet@miller...|       2020-06-17|https://reilly-fr...|
|  124|8CaB7c7cbe6da4f|     Casey|   Conway|Henderson, Ruiz a...|   Lake Josemouth|             Ukraine|        725.960.3587|         6110447130|jesuswoodard@brig...|       202

+-----------+-----+
|    Country|count|
+-----------+-----+
|      Spain|    3|
|   Kiribati|    2|
|Puerto Rico|    2|
+-----------+-----+
only showing top 3 rows



In [5]:
## Working with numbers

from pyspark.sql.functions import expr, pow, round, bround, lit, corr, monotonically_increasing_id

powered_plus_one_index = pow(col("Index"), 2) + 1
df.select( expr("index"), powered_plus_one_index.alias("new_index") ).show(5)
df.selectExpr( "index", "POWER(index, 2)+1 as new_index" ).show(5)

df.select( round(lit(2.5)), bround(lit(2.5)) ).show(3)

df.selectExpr( "index", "POWER(index, 2)+1 as new_index" ).select( corr("index", "new_index") ).show()

df.limit(10000).select("Index", "First Name", "Company", "City", "Country", "Email", "Subscription Date").describe().show()

df.stat.approxQuantile("Index", [0.5], 0.05)

# df.stat.crosstab("index", "Country").show()
df.stat.freqItems(['index', 'Country']).show()

df.select(monotonically_increasing_id()).show(2)

+-----+---------+
|index|new_index|
+-----+---------+
|    1|      2.0|
|    2|      5.0|
|    3|     10.0|
|    4|     17.0|
|    5|     26.0|
+-----+---------+
only showing top 5 rows

+-----+---------+
|index|new_index|
+-----+---------+
|    1|      2.0|
|    2|      5.0|
|    3|     10.0|
|    4|     17.0|
|    5|     26.0|
+-----+---------+
only showing top 5 rows

+-------------+--------------+
|round(2.5, 0)|bround(2.5, 0)|
+-------------+--------------+
|          3.0|           2.0|
|          3.0|           2.0|
|          3.0|           2.0|
+-------------+--------------+
only showing top 3 rows



+----------------------+
|corr(index, new_index)|
+----------------------+
|     0.968246441709095|
+----------------------+



26/06/01 14:28:17 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-------+------------------+----------+--------------+---------+-----------+--------------------+
|summary|             Index|First Name|       Company|     City|    Country|               Email|
+-------+------------------+----------+--------------+---------+-----------+--------------------+
|  count|             10000|     10000|         10000|    10000|      10000|               10000|
|   mean|            5000.5|      null|          null|     null|       null|                null|
| stddev|2886.8956799071675|      null|          null|     null|       null|                null|
|    min|                 1|     Aaron|    Abbott Inc|Aaronberg|Afghanistan| aabbott@camacho.com|
|    max|             10000|       Zoe|Zuniga-Whitney|  Zoeport|   Zimbabwe|zzhang@osborn-bri...|
+-------+------------------+----------+--------------+---------+-----------+--------------------+



+--------------------+--------------------+
|     index_freqItems|   Country_freqItems|
+--------------------+--------------------+
|[99991, 99946, 99...|[Guinea, Guinea-B...|
+--------------------+--------------------+

+-----------------------------+
|monotonically_increasing_id()|
+-----------------------------+
|                            0|
|                            1|
+-----------------------------+
only showing top 2 rows



In [6]:
## Working with Strings
from pyspark.sql.functions import col, initcap, lit, lower, upper, ltrim, rtrim, trim, lpad, rpad

df = df.limit(10000).select("Index", "Customer Id", "First Name", "Last Name", "Company", "City", "Country", "Email", "Subscription Date")

df.select( "*", initcap(lit('example text unimportant')).alias('initCapField') ).show(10)

df.select( col('Company'), lower(col('Company')), upper(col('Company')) ).show(5)

df.select(
    ltrim(lit("   HELLO   ")).alias("ltrim"),
    rtrim(lit("   HELLO   ")).alias("rtrim"),
    trim(lit("   HELLO   ")).alias("trim"),
    lpad(lit("HELLO"), 3, " ").alias("lpad"),
    rpad(lit("HELLO"), 10, " ").alias("rpad")
).show(1)

+-----+---------------+----------+----------+--------------------+--------------+--------------------+--------------------+-----------------+--------------------+
|Index|    Customer Id|First Name| Last Name|             Company|          City|             Country|               Email|Subscription Date|        initCapField|
+-----+---------------+----------+----------+--------------------+--------------+--------------------+--------------------+-----------------+--------------------+
|    1|ffeCAb7AbcB0f07|     Jared|    Jarvis|    Sanchez-Fletcher| Hatfieldshire|             Eritrea|gabriellehartman@...|       2021-11-11|Example Text Unim...|
|    2|b687FfC4F1600eC|     Marie|    Malone|           Mckay PLC|Robertsonburgh|            Botswana|kstafford@sexton.com|       2021-05-14|Example Text Unim...|
|    3|9FF9ACbc69dcF9c|    Elijah|   Barrera|      Marks and Sons|       Kimbury|            Barbados|jeanettecross@bro...|       2021-03-17|Example Text Unim...|
|    4|b49edDB1295FF6E

In [7]:
## Regex Expressions

from pyspark.sql.functions import regexp_replace, translate, regexp_extract, instr, locate

regex_string = "Ja|ja|Je|je"
df.select(
    regexp_replace( col("First Name"), regex_string, "GA").alias("JChanged"),
    col("First Name")
).show(5)

df.select( col("First Name"), translate(col("First Name"), "Ja", "GA" ).alias("Transformed") ).show(3)

extract_str = "(JA|ja|JE|je)"
df.select(
    regexp_extract( col('First Name'), extract_str, 1 ).alias("Transformed"),
    col("First Name")
).show(5)

containsJa = instr( col('First Name'), 'Ja' ) >= 1
containsJe = instr( col('First Name'), "Je" ) >= 1

df.withColumn( "JaJe", containsJa | containsJe ).where('JaJe').select('First Name').show(10)

simpleSyllabes = ['Ja', 'Je']
def syllabe_locator(column, syllabe_string):
    return locate(syllabe_string, column).cast('boolean').alias('is_' + syllabe_string)

selectedColumns = [syllabe_locator(df['First Name'], c) for c in simpleSyllabes]
selectedColumns.append( expr("*") )

df.select(*selectedColumns).where( expr("is_ja or is_je") ).select("First Name").show(3)

+--------+----------+
|JChanged|First Name|
+--------+----------+
|   GAred|     Jared|
|   Marie|     Marie|
|  EliGAh|    Elijah|
|  Sheryl|    Sheryl|
|  GAremy|    Jeremy|
+--------+----------+
only showing top 5 rows

+----------+-----------+
|First Name|Transformed|
+----------+-----------+
|     Jared|      GAred|
|     Marie|      MArie|
|    Elijah|     ElijAh|
+----------+-----------+
only showing top 3 rows

+-----------+----------+
|Transformed|First Name|
+-----------+----------+
|           |     Jared|
|           |     Marie|
|         ja|    Elijah|
|           |    Sheryl|
|           |    Jeremy|
+-----------+----------+
only showing top 5 rows



+----------+
|First Name|
+----------+
|     Jared|
|    Jeremy|
|     Jamie|
|    Jackie|
|     James|
|  Jennifer|
|   Jessica|
|     Jesse|
|   Jeffery|
|    Jackie|
+----------+
only showing top 10 rows

+----------+
|First Name|
+----------+
|     Jared|
|    Jeremy|
|     Jamie|
+----------+
only showing top 3 rows



In [8]:
## Working with Dates and Timestamps

from pyspark.sql.functions import current_date, current_timestamp

daf = spark.range(10).withColumn("today", current_date()).withColumn("now", current_timestamp())
daf.show()

+---+----------+--------------------+
| id|     today|                 now|
+---+----------+--------------------+
|  0|2026-06-01|2026-06-01 14:28:...|
|  1|2026-06-01|2026-06-01 14:28:...|
|  2|2026-06-01|2026-06-01 14:28:...|
|  3|2026-06-01|2026-06-01 14:28:...|
|  4|2026-06-01|2026-06-01 14:28:...|
|  5|2026-06-01|2026-06-01 14:28:...|
|  6|2026-06-01|2026-06-01 14:28:...|
|  7|2026-06-01|2026-06-01 14:28:...|
|  8|2026-06-01|2026-06-01 14:28:...|
|  9|2026-06-01|2026-06-01 14:28:...|
+---+----------+--------------------+



In [9]:
from pyspark.sql.functions import date_add, date_sub

daf.select( date_sub(col("today"), 5), date_add(col("today"), 5) ).show(1)

+------------------+------------------+
|date_sub(today, 5)|date_add(today, 5)|
+------------------+------------------+
|        2026-05-27|        2026-06-06|
+------------------+------------------+
only showing top 1 row



In [10]:
from pyspark.sql.functions import datediff, months_between, to_date

daf.withColumn("week_ago", date_sub(col("today"), 7)).select( datediff(col("week_ago"), col("today")) ).show(1)
daf.select(
    to_date( lit("2016-01-01") ).alias("start"),
    to_date( lit("2017-05-22") ).alias("end")
).select( months_between(col("start"), col("end")) ).show(1)

+-------------------------+
|datediff(week_ago, today)|
+-------------------------+
|                       -7|
+-------------------------+
only showing top 1 row

+--------------------------------+
|months_between(start, end, true)|
+--------------------------------+
|                    -16.67741935|
+--------------------------------+
only showing top 1 row



In [11]:
from pyspark.sql.functions import to_date, lit, to_timestamp

date = spark.range(5).withColumn("date", lit("2017-01-01")).select( to_date(col("date")) )
date.show(1)
date.select( to_date(lit("2016-20-12")), to_date(lit("2017-12-11")) ).show(1)

dateFormat = "yyyy-dd-MM"
cleanDateDf = spark.range(1).select(
    to_date( lit("2017-12-11"), dateFormat ).alias("date"),
    to_date( lit("2017-20-12"), dateFormat ).alias("date2")
)
cleanDateDf.show()

cleanDateDf.select( to_timestamp(col("date"), dateFormat) ).show()

+-------------+
|to_date(date)|
+-------------+
|   2017-01-01|
+-------------+
only showing top 1 row

+-------------------+-------------------+
|to_date(2016-20-12)|to_date(2017-12-11)|
+-------------------+-------------------+
|               null|         2017-12-11|
+-------------------+-------------------+
only showing top 1 row

+----------+----------+
|      date|     date2|
+----------+----------+
|2017-11-12|2017-12-20|
+----------+----------+

+------------------------------+
|to_timestamp(date, yyyy-dd-MM)|
+------------------------------+
|           2017-11-12 00:00:00|
+------------------------------+



In [12]:
cleanDateDf.show()
cleanDateDf.filter( col("date2") > lit("2017-12-12") ).show()
cleanDateDf.filter( col("date2") > "'2017-12-12'" ).show()

+----------+----------+
|      date|     date2|
+----------+----------+
|2017-11-12|2017-12-20|
+----------+----------+

+----------+----------+
|      date|     date2|
+----------+----------+
|2017-11-12|2017-12-20|
+----------+----------+

+----+-----+
|date|date2|
+----+-----+
+----+-----+



In [15]:
## Working with Nulls in Data

cleanDateDf.na.fill(2).show()

### Coalesce

from pyspark.sql.functions import coalesce

df.show()
df.select( coalesce( col("Customer Id"), col("First Name") ).alias("Coalesce") ).show()

+----------+----------+
|      date|     date2|
+----------+----------+
|2017-11-12|2017-12-20|
+----------+----------+

+-----+---------------+----------+----------+--------------------+-----------------+--------------------+--------------------+-----------------+
|Index|    Customer Id|First Name| Last Name|             Company|             City|             Country|               Email|Subscription Date|
+-----+---------------+----------+----------+--------------------+-----------------+--------------------+--------------------+-----------------+
|    1|ffeCAb7AbcB0f07|     Jared|    Jarvis|    Sanchez-Fletcher|    Hatfieldshire|             Eritrea|gabriellehartman@...|       2021-11-11|
|    2|b687FfC4F1600eC|     Marie|    Malone|           Mckay PLC|   Robertsonburgh|            Botswana|kstafford@sexton.com|       2021-05-14|
|    3|9FF9ACbc69dcF9c|    Elijah|   Barrera|      Marks and Sons|          Kimbury|            Barbados|jeanettecross@bro...|       2021-03-17|
|    4|b4

In [38]:
## ifnull, nullif, nvl, and nvl2

"""
from pyspark.sql.functions import ifnull, nullif, nvl, nvl2, col 

df.select(
    col("Index"),
    col("City"),
    ifnull(col("Index"), col("City")).alias("ifnull_res"),
    nvl(col("Index"), col("City")).alias("nvl_res"),
    nvl2(col("Index"), col("Email"), col("City")).alias("nvl2_res"),
    nullif(col("Index"), col("City")).alias("nullif_res")
).show()
"""

'\nfrom pyspark.sql.functions import ifnull, nullif, nvl, nvl2, col \n\ndf.select(\n    col("Index"),\n    col("City"),\n    ifnull(col("Index"), col("City")).alias("ifnull_res"),\n    nvl(col("Index"), col("City")).alias("nvl_res"),\n    nvl2(col("Index"), col("Email"), col("City")).alias("nvl2_res"),\n    nullif(col("Index"), col("City")).alias("nullif_res")\n).show()\n'

In [58]:
## drop

df.drop("Company").drop("City").show(5)
df.na.drop().show(5)
df.na.drop("any").show(5)
df.na.drop("all").show(5)
df.na.drop("all", subset=["Company", "Country"]).show(5)

+-----+---------------+----------+----------+--------------------+--------------------+-----------------+
|Index|    Customer Id|First Name| Last Name|             Country|               Email|Subscription Date|
+-----+---------------+----------+----------+--------------------+--------------------+-----------------+
|    1|ffeCAb7AbcB0f07|     Jared|    Jarvis|             Eritrea|gabriellehartman@...|       2021-11-11|
|    2|b687FfC4F1600eC|     Marie|    Malone|            Botswana|kstafford@sexton.com|       2021-05-14|
|    3|9FF9ACbc69dcF9c|    Elijah|   Barrera|            Barbados|jeanettecross@bro...|       2021-03-17|
|    4|b49edDB1295FF6E|    Sheryl|Montgomery|Antarctica (the t...|thomassierra@barr...|       2020-09-23|
|    5|3dcCbFEB17CCf2E|    Jeremy|   Houston|          Micronesia|rubenwatkins@jaco...|       2020-09-18|
+-----+---------------+----------+----------+--------------------+--------------------+-----------------+
only showing top 5 rows

+-----+--------------

+-----+---------------+----------+----------+--------------------+--------------+--------------------+--------------------+-----------------+
|Index|    Customer Id|First Name| Last Name|             Company|          City|             Country|               Email|Subscription Date|
+-----+---------------+----------+----------+--------------------+--------------+--------------------+--------------------+-----------------+
|    1|ffeCAb7AbcB0f07|     Jared|    Jarvis|    Sanchez-Fletcher| Hatfieldshire|             Eritrea|gabriellehartman@...|       2021-11-11|
|    2|b687FfC4F1600eC|     Marie|    Malone|           Mckay PLC|Robertsonburgh|            Botswana|kstafford@sexton.com|       2021-05-14|
|    3|9FF9ACbc69dcF9c|    Elijah|   Barrera|      Marks and Sons|       Kimbury|            Barbados|jeanettecross@bro...|       2021-03-17|
|    4|b49edDB1295FF6E|    Sheryl|Montgomery|Kirby, Vaughn and...|   Briannaview|Antarctica (the t...|thomassierra@barr...|       2020-09-23|
|    5

In [62]:
# fill

df.show(5)

df.na.fill("All Null values become this string").show(5)
df.na.fill("all", subset=["Company", "Country"]).show(5)
df.na.replace(["Marks and Sons"], ["Unknown"], "Company").show(5)

+-----+---------------+----------+----------+--------------------+--------------+--------------------+--------------------+-----------------+
|Index|    Customer Id|First Name| Last Name|             Company|          City|             Country|               Email|Subscription Date|
+-----+---------------+----------+----------+--------------------+--------------+--------------------+--------------------+-----------------+
|    1|ffeCAb7AbcB0f07|     Jared|    Jarvis|    Sanchez-Fletcher| Hatfieldshire|             Eritrea|gabriellehartman@...|       2021-11-11|
|    2|b687FfC4F1600eC|     Marie|    Malone|           Mckay PLC|Robertsonburgh|            Botswana|kstafford@sexton.com|       2021-05-14|
|    3|9FF9ACbc69dcF9c|    Elijah|   Barrera|      Marks and Sons|       Kimbury|            Barbados|jeanettecross@bro...|       2021-03-17|
|    4|b49edDB1295FF6E|    Sheryl|Montgomery|Kirby, Vaughn and...|   Briannaview|Antarctica (the t...|thomassierra@barr...|       2020-09-23|
|    5

In [80]:
## Structs

print( df.selectExpr("(Company, City) as complex", "*") )
print( df.selectExpr("struct(Company, City) as complex", "*") )

from pyspark.sql.functions import struct

complexDF = df.select( struct("Company", "City").alias("complex") )
complexDF.createOrReplaceTempView("complexDF")

complexDF.show(5)
complexDF.select("complex.Company").show(5)
complexDF.select( col("complex").getField("Company") ).show(5)
complexDF.select("complex.*").show(5)

DataFrame[complex: struct<Company:string,City:string>, Index: int, Customer Id: string, First Name: string, Last Name: string, Company: string, City: string, Country: string, Email: string, Subscription Date: date]
DataFrame[complex: struct<Company:string,City:string>, Index: int, Customer Id: string, First Name: string, Last Name: string, Company: string, City: string, Country: string, Email: string, Subscription Date: date]
+--------------------+
|             complex|
+--------------------+
|{Sanchez-Fletcher...|
|{Mckay PLC, Rober...|
|{Marks and Sons, ...|
|{Kirby, Vaughn an...|
|{Lester-Manning, ...|
+--------------------+
only showing top 5 rows

+--------------------+
|             Company|
+--------------------+
|    Sanchez-Fletcher|
|           Mckay PLC|
|      Marks and Sons|
|Kirby, Vaughn and...|
|      Lester-Manning|
+--------------------+
only showing top 5 rows

+--------------------+
|     complex.Company|
+--------------------+
|    Sanchez-Fletcher|
|           Mc

In [86]:
## split

from pyspark.sql.functions import split

df.select(split(col("Company"), " ").alias("array_col")).selectExpr("array_col[0]").show(5)

+----------------+
|    array_col[0]|
+----------------+
|Sanchez-Fletcher|
|           Mckay|
|           Marks|
|          Kirby,|
|  Lester-Manning|
+----------------+
only showing top 5 rows



In [90]:
## Array Length
from pyspark.sql.functions import size

df.select( size( split( col("Company"), " " ) ) ).show(5)

+---------------------------+
|size(split(Company,  , -1))|
+---------------------------+
|                          1|
|                          2|
|                          3|
|                          4|
|                          1|
+---------------------------+
only showing top 5 rows



In [95]:
## Array_contains
from pyspark.sql.functions import array_contains

df.select( array_contains( split( col("Company"), " " ), "Marks" ) ).show(5)

+--------------------------------------------+
|array_contains(split(Company,  , -1), Marks)|
+--------------------------------------------+
|                                       false|
|                                       false|
|                                        true|
|                                       false|
|                                       false|
+--------------------------------------------+
only showing top 5 rows



In [107]:
## Explode
from pyspark.sql.functions import col, split, explode

df.withColumn( "splitted", split( col("Company"), " " ) )\
    .withColumn("exploded", explode( col("splitted") ) )\
    .select("Company", "splitted", "exploded").show(5)

+----------------+------------------+----------------+
|         Company|          splitted|        exploded|
+----------------+------------------+----------------+
|Sanchez-Fletcher|[Sanchez-Fletcher]|Sanchez-Fletcher|
|       Mckay PLC|      [Mckay, PLC]|           Mckay|
|       Mckay PLC|      [Mckay, PLC]|             PLC|
|  Marks and Sons|[Marks, and, Sons]|           Marks|
|  Marks and Sons|[Marks, and, Sons]|             and|
+----------------+------------------+----------------+
only showing top 5 rows



In [128]:
## Maps

from pyspark.sql.functions import create_map

df_map = df.select( 
    create_map( col("Company"), col("City") ).alias("complex_map") 
)
df_map.show(3)
df_map.selectExpr("complex_map['Mckay PLC']").show(3)

# df.select( map( col("Company"), col("City") ).alias("complex_map") ).selectExpr("explor(complex_map)]").show(2)

+--------------------+
|         complex_map|
+--------------------+
|{Sanchez-Fletcher...|
|{Mckay PLC -> Rob...|
|{Marks and Sons -...|
+--------------------+
only showing top 3 rows

+----------------------+
|complex_map[Mckay PLC]|
+----------------------+
|                  null|
|        Robertsonburgh|
|                  null|
+----------------------+
only showing top 3 rows



In [139]:
## Json

jsondf = spark.range(1).selectExpr("""
'{"myJSONKey" : { "myJSONValue" : [1,2,3]}}' as jsonString
""")

In [155]:
from pyspark.sql.functions import get_json_object, json_tuple

jsondf.select(
    get_json_object( col("jsonString"), "$.myJSONKey.myJSONValue[1]" ).alias("column"),
    json_tuple( col("jsonString"), "myJSONKey" )
).show(2)

+------+--------------------+
|column|                  c0|
+------+--------------------+
|     2|{"myJSONValue":[1...|
+------+--------------------+



In [171]:
## User defined functions

udfExampleDF = spark.range(5).toDF("num")

def power3(double_value):
        return double_value ** 3

power3(2.0)

from pyspark.sql.functions import udf, col
from pyspark.sql.types import IntegerType

power3udf = udf(power3)
udfExampleDF.select( power3udf(col("num")) ).show()

spark.udf.register("power3", power3, IntegerType())
udfExampleDF.selectExpr("power3(num)").show()

+-----------+
|power3(num)|
+-----------+
|          0|
|          1|
|          8|
|         27|
|         64|
+-----------+



26/06/01 16:36:00 WARN SimpleFunctionRegistry: The function power3 replaced a previously registered function.
[Stage 219:============================>                           (6 + 6) / 12]

+-----------+
|power3(num)|
+-----------+
|          0|
|          1|
|          8|
|         27|
|         64|
+-----------+

